## NOTE:
This is a PROVIDER_ID based version of code. Confirm if you need NPI or PROVIDER_ID

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_step23_function"

In [0]:
import gc

### HCP Universe

In [0]:
hcp_universe = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/kerendia_nba_3_hcp_model_base_hcp_universe_for_inference")
hcp_universe = hcp_universe.select("PROVIDER_ID")

print("Shape :", hcp_universe.count(), len(hcp_universe.columns))
print("# Distinct HCPs :", hcp_universe.select("PROVIDER_ID").distinct().count())

### Load Data

In [0]:
cohort_anchor_date = spark.read.csv("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/hcp_model_inference_anchor_dates.csv", header = True, inferSchema = True)
LOOKBACK_START_DATE = cohort_anchor_date.filter(F.col("COHORT") == "I01").first()["LOOKBACK_START_DATE"]
LOOKBACK_START_DATE_STR = LOOKBACK_START_DATE.strftime("%Y%m%d")
print(LOOKBACK_START_DATE_STR)
LOOKBACK_END_DATE = cohort_anchor_date.filter(F.col("COHORT") == "I01").first()["LOOKBACK_END_DATE"]
cohort_anchor_date.display()

In [0]:
# LAAD-based LRx
query = """
            SELECT DISTINCT WEEK_ENDING_DATE, PROVIDER_ID, PERIOD_ID, PERIOD_TYPE, SPECIALTY_GROUP, PRODUCT_GROUP, SWITCH_FROM, ADD_ON_TO, AGE_GROUP, GENDER, PAY_TYPE, SOURCE_OF_BUSINESS, RX_CHANNEL, INDICATION_GROUP, HYPERTENSION_FLAG, PAYER_PLAN_ID, NBRX, TRX, NRX, NPI, PRI_SPCL_DESC
            from `ph-com-ai-us-prod-catalog-2553213839599380`.`kerendia_laad_based_lrx`.KERENDIA_LAAD_BASED_LRX_FULL_DATA_STARTING_2021
            WHERE WEEK_ENDING_DATE < '20230101'
      """

hcp_lrx_claims_old = spark.sql(query)
print("Shape :", hcp_lrx_claims_old.count(), len(hcp_lrx_claims_old.columns))

In [0]:
# LAAD-based LRx
query = """
            SELECT DISTINCT WEEK_ENDING_DATE, PROVIDER_ID, PERIOD_ID, PERIOD_TYPE, SPECIALTY_GROUP, PRODUCT_GROUP, SWITCH_FROM, ADD_ON_TO, AGE_GROUP, GENDER, PAY_TYPE, SOURCE_OF_BUSINESS, RX_CHANNEL, INDICATION_GROUP, HYPERTENSION_FLAG, PAYER_PLAN_ID, NBRX, TRX, NRX, NPI, PRI_SPCL_DESC
            from `ph-com-ai-us-prod-catalog-2553213839599380`.`kerendia_laad_based_lrx`.KERENDIA_LAAD_BASED_LRX_V6
            WHERE WEEK_ENDING_DATE >= '20230101'
      """

hcp_lrx_claims_new = spark.sql(query)
print("Shape :", hcp_lrx_claims_new.count(), len(hcp_lrx_claims_new.columns))

In [0]:
hcp_lrx_claims_new = hcp_lrx_claims_new.withColumn(
    "PERIOD_ID", date_format("PERIOD_ID", "yyyyMMdd")
)
hcp_lrx_claims_new.limit(5).display()

In [0]:
hcp_lrx_claims = hcp_lrx_claims_new.union(hcp_lrx_claims_old)
# hcp_lrx_claims = hcp_lrx_claims_new

In [0]:
# Filtering LRx claims only for HCPs of interest
hcp_lrx_claims = hcp_universe.join(hcp_lrx_claims, on = "PROVIDER_ID",how = "inner")
print("Shape :", hcp_lrx_claims.count(), len(hcp_lrx_claims.columns))
print("# Distinct Providers :", hcp_lrx_claims.select("PROVIDER_ID").distinct().count())

In [0]:
hcp_lrx_claims = hcp_lrx_claims.withColumn("INDICATION_GROUP", F.when(col("INDICATION_GROUP") == "ALL OTHERS", "ALL_OTHERS").otherwise(col("INDICATION_GROUP")))

In [0]:
hcp_lrx_claims = hcp_lrx_claims.withColumn("SVC_DT",concat(substring('PERIOD_ID',1,4),substring('PERIOD_ID',5,2),substring('PERIOD_ID',7,2)))

hcp_lrx_claims = hcp_lrx_claims.withColumn("YEAR_MONTH_DAY",F.to_date(F.unix_timestamp(hcp_lrx_claims.SVC_DT,'yyyyMMdd').cast('timestamp')))

In [0]:
# Truncate date to year month level
hcp_lrx_claims = hcp_lrx_claims.withColumn("SVC_DT_M",trunc("YEAR_MONTH_DAY",'month'))

# Drop unnecessary columns 
hcp_lrx_claims = hcp_lrx_claims.drop('SVC_DT').drop('YEAR_MONTH_DAY')

hcp_lrx_claims.limit(10).display()

## Create Features

In [0]:
index_cols = ["PROVIDER_ID", "SVC_DT_M"]

###1.1 Product Group

In [0]:
# Remove null values from Drug class column
df_category_claims = (
  hcp_lrx_claims
  .withColumn("PRODUCT_GROUP",
                F.when(((col("PRODUCT_GROUP").isNull()) | (col('PRODUCT_GROUP') == 'NULL')), 'UNKNOWN_PG').otherwise(col('PRODUCT_GROUP')))
  # .filter("DAYS_SUPPLY <> 'ignore'")  
)

# Drug category
df_category = (
  df_category_claims
  .groupBy(index_cols).pivot("PRODUCT_GROUP")
  .agg(
    F.sum("TRX").alias("SUM_TRX")
    )
  .na.fill(value=0)
)

# clean column name
df_category = df_category.select([F.col(c).alias('TOTAL_HCP_TRX_CLAIMS_PRODUCT_GROUP_' + c.replace(", ", " ").replace("+", "and").replace("/", " or ").replace("-", "_").replace(" ", "_").replace("__", "_").replace("(", "").replace(")", "").replace("&", "").upper()) if c not in index_cols else col(c) for c in df_category.columns])

###1.2 Specialty Group

In [0]:
# Remove null values from Drug class column
df_specialty_grp_claims = (
  hcp_lrx_claims
  .withColumn("SPECIALTY_GROUP",
                F.when(((col("SPECIALTY_GROUP").isNull()) | (col('SPECIALTY_GROUP') == 'NULL')), 'UNKNOWN').otherwise(col('SPECIALTY_GROUP')))
  # .filter("DAYS_SUPPLY <> 'ignore'")  
)

# Drug category
df_specialty = (
  df_specialty_grp_claims
  .groupBy(index_cols).pivot("SPECIALTY_GROUP")
  .agg(
    F.sum("TRX").alias("SUM_TRX")
    )
  .na.fill(value=0)
)

# clean column name
df_specialty = df_specialty.select([F.col(c).alias('TOTAL_HCP_TRX_CLAIMS_SPEC_' + c.replace(", ", " ").replace("+", "and").replace("/", " or ").replace("-", "_").replace(" ", "_").replace("__", "_").replace("(", "").replace(")", "").replace("&", "").upper()) if c not in index_cols else col(c) for c in df_specialty.columns])


###1.3 Switch From

In [0]:
# Remove null values from Drug class column
df_switch_frm_claims = (
  hcp_lrx_claims
  .withColumn("SWITCH_FROM",
                F.when(((col("SWITCH_FROM").isNull()) | (col('SWITCH_FROM') == 'NULL')), 'UNKNOWN').otherwise(col('SWITCH_FROM')))
  # .filter("DAYS_SUPPLY <> 'ignore'")  
)

# Drug category
df_switch = (
  df_switch_frm_claims
  .groupBy(index_cols).pivot("SWITCH_FROM")
  .agg(
    F.sum("TRX").alias("SUM_TRX")
    )
  .na.fill(value=0)
)

# clean column name
df_switch = df_switch.select([F.col(c).alias('TOTAL_HCP_TRX_CLAIMS_SWITCH_FROM_' + c.replace(", ", " ").replace("+", "and").replace("/", " or ").replace("-", "_").replace(" ", "_").replace("__", "_").replace("(", "").replace(")", "").replace("&", "").upper()) if c not in index_cols else col(c) for c in df_switch.columns])

###1.4 Add on to

In [0]:
# Remove null values from Drug class column
df_add_on_claims = (
  hcp_lrx_claims
  .withColumn("ADD_ON_TO",
                F.when(((col("ADD_ON_TO").isNull()) | (col('ADD_ON_TO') == 'NULL')), 'UNKNOWN').otherwise(col('ADD_ON_TO')))
  # .filter("DAYS_SUPPLY <> 'ignore'")  
)

# Drug category
df_add_on_to = (
  df_add_on_claims
  .groupBy(index_cols).pivot("ADD_ON_TO")
  .agg(
    F.sum("TRX").alias("SUM_TRX")
    )
  .na.fill(value=0)
)

# clean column name
df_add_on_to = df_add_on_to.select([F.col(c).alias('TOTAL_HCP_TRX_CLAIMS_ADD_ON_TO_' + c.replace(", ", " ").replace("+", "and").replace("/", " or ").replace("-", "_").replace(" ", "_").replace("__", "_").replace("(", "").replace(")", "").replace("&", "").upper()) if c not in index_cols else col(c) for c in df_add_on_to.columns])


###1.5 Age Group

In [0]:
# age group
df_patient_age = (
  hcp_lrx_claims
  .groupBy(index_cols).pivot("AGE_GROUP")
  .agg(
    F.sum("TRX").alias("SUM_TRX")
    )

  .na.fill(value=0)
)

# clean column name
df_patient_age = df_patient_age.select([F.col(c).alias('TOTAL_HCP_TRX_CLAIMS_AGE_' + c.replace(", ", " ").replace("+", "above").replace("/", " or ").replace("-", "_").replace("<", "LESS_THAN_").replace(" ", "_").replace("__", "_").replace("(", "").replace(")", "").replace("&", "").upper()) if c not in index_cols else col(c) for c in df_patient_age.columns])

###1.6 Pay Type

In [0]:
# Join the dataframes 
df_payer = (
  hcp_lrx_claims
  .groupBy(index_cols).pivot("PAY_TYPE")
  .agg(
    F.sum("TRX").alias("SUM_TRX")
    )
  .na.fill(value=0)
)


# clean column name
df_payer = df_payer.select([F.col(c).alias('TOTAL_HCP_TRX_CLAIMS_PAY_TYPE_' + c.replace(", ", " ").replace("+", "and").replace("/", " or ").replace("-", "_").replace(" ", "_").replace("__", "_").replace("(", "").replace(")", "").replace("&", "").upper()) if c not in index_cols else col(c) for c in df_payer.columns])

###1.6 Source Of Business

In [0]:
# Drug brand
df_sob = (
  hcp_lrx_claims
  .groupBy(index_cols).pivot("SOURCE_OF_BUSINESS")
  .agg(
    F.sum("TRX").alias("SUM_TRX")
    )
  .na.fill(value=0)
)

# clean column name
df_sob = df_sob.select([F.col(c).alias('TOTAL_HCP_TRX_CLAIMS_SOB_' + c.replace(", ", " ").replace("+", "and").replace("/", " or ").replace("-", "_").replace(" ", "_").replace("__", "_").replace("(", "").replace(")", "").replace("&", "").upper()) if c not in index_cols else col(c) for c in df_sob.columns])


###1.7 Channel Level

In [0]:
# Drug brand
df_channels = (
  hcp_lrx_claims
  .groupBy(index_cols).pivot("RX_CHANNEL")
  .agg(
    F.sum("TRX").alias("SUM_TRX")
    )
  .na.fill(value=0)
)

# clean column name
df_channels = df_channels.select([F.col(c).alias('TOTAL_HCP_TRX_CLAIMS_CHANNEL_' + c.replace(", ", " ").replace("+", "and").replace("/", " or ").replace("-", "_").replace(" ", "_").replace("__", "_").replace("(", "").replace(")", "").replace("&", "").upper()) if c not in index_cols else col(c) for c in df_channels.columns])

###1.8 Indication Group

In [0]:
# Join the dataframes 
df_indication_grp = (
  hcp_lrx_claims
  .groupBy(index_cols).pivot("INDICATION_GROUP")
  .agg(
    F.sum("TRX").alias("SUM_TRX")
    )
  .na.fill(value=0)
)


# clean column name
df_indication_grp = df_indication_grp.select([F.col(c).alias('TOTAL_HCP_TRX_CLAIMS_INDICATION_GROUP_' + c.replace(", ", " ").replace("+", "and").replace("/", " or ").replace("-", "_").replace(" ", "_").replace("__", "_").replace("(", "").replace(")", "").replace("&", "").upper()) if c not in index_cols else col(c) for c in df_indication_grp.columns])


###1.9 Hypertension Flag

In [0]:
# Remove null values from hypertension column
df_hypertension_claims = (
  hcp_lrx_claims
  .withColumn("HYPERTENSION_FLAG",
                F.when(((col("HYPERTENSION_FLAG").isNull()) | (col('HYPERTENSION_FLAG') == 'NULL')), 'UNKNOWN_HP').otherwise(col('HYPERTENSION_FLAG')))
  # .filter("DAYS_SUPPLY <> 'ignore'")  
)

df_hypertension_flag = (
  df_hypertension_claims
  .groupBy(index_cols).pivot("HYPERTENSION_FLAG")
  .agg(
    F.sum("TRX").alias("SUM_TRX")
    )

  .na.fill(value=0)
)

# clean column name
df_hypertension_flag = df_hypertension_flag.select([F.col(c).alias('TOTAL_HCP_TRX_CLAIMS_HYPERTENSION_' + c.replace(", ", " ").replace("+", "and").replace("/", " or ").replace("-", "_").replace(" ", "_").replace("__", "_").replace("(", "").replace(")", "").replace("&", "").upper()) if c not in index_cols else col(c) for c in df_hypertension_flag.columns])


In [0]:
# Remove null values from column
df_gender_claims = (
  hcp_lrx_claims
  .withColumn("GENDER",
                F.when(((col("GENDER").isNull()) | (col('GENDER') == 'NULL')), 'UNKNOWN').otherwise(col('GENDER')))
  # .filter("DAYS_SUPPLY <> 'ignore'")  
)

# gender category
df_gender = (
  df_gender_claims
  .groupBy(index_cols).pivot("GENDER")
  .agg(
    F.sum("TRX").alias("SUM_TRX")
    )
  .na.fill(value=0)
)

# clean column name
df_gender = df_gender.select([F.col(c).alias('TOTAL_HCP_TRX_CLAIMS_GENDER_' + c.replace(", ", " ").replace("+", "and").replace("/", " or ").replace("-", "_").replace(" ", "_").replace("__", "_").replace("(", "").replace(")", "").replace("&", "").upper()) if c not in index_cols else col(c) for c in df_gender.columns])

In [0]:
# combine features
df_spine = (
  df_category
  .join(df_specialty, how = 'left', on = index_cols)
  .join(df_switch, how = "left", on = index_cols)
  .join(df_add_on_to, how = "left", on = index_cols)
  .join(df_patient_age, how = 'left', on = index_cols)
  .join(df_payer, how = 'left', on = index_cols)
  .join(df_sob, how = 'left', on = index_cols)
  .join(df_channels, how = "left", on = index_cols)
  .join(df_indication_grp, how = "left", on = index_cols)
  .join(df_hypertension_flag, how = 'left', on = index_cols)
  .join(df_gender, how = "left", on = index_cols))

In [0]:
del df_specialty, df_switch, df_add_on_to, df_patient_age, df_payer, df_sob, df_channels, df_indication_grp, df_hypertension_flag, df_gender # , df_pri_specialty
gc.collect()

##3. Fill Missing Months

In [0]:
df_spine_step2 = step2_fill_missing(df_spine, key_col="PROVIDER_ID", date_col="SVC_DT_M")

In [0]:
##  list of col names
keep_cols = ["PROVIDER_ID", "SVC_DT_M"]
cols_to_drop = [col_name for col_name in df_spine_step2.columns if col_name not in keep_cols]

##4. Create Lag Features

In [0]:
# UPDATE - 2024-05-20 NBA 2.3 ENHANCEMENT
# FEATURES BEING CREATED WITH 12 MONTH LAG ONLY, 18, 24 MONTH LAG FEATURES HAVE BEEN ELIMINATED DUE TO DATA UNAVAILABILITY
# last_i_month_list = [1, 3, 6, 12, 18, 24]

last_i_month_list = [1, 3, 6, 12]

In [0]:
df_trx_feats = step3_lag(
  df_spine_step2, 
  key_col="PROVIDER_ID", date_col="SVC_DT_M", 
  last_i_month_list=last_i_month_list
)

In [0]:
df_trx_feats = df_trx_feats.drop(*cols_to_drop)

In [0]:
# Remove null values from NPI Number column
df_trx_feats = df_trx_feats.filter(df_trx_feats.PROVIDER_ID.isNotNull())
df_trx_feats = df_trx_feats.withColumnRenamed("NPI","NPI_NUMBER")

In [0]:
# @change -> Need to update this date
df_trx_feats = df_trx_feats.filter(F.to_date('SVC_DT_M') >= '2023-01-01')

### 5. Write Data

In [0]:
print("Shape :", df_trx_feats.count(), len(df_trx_feats.columns))

In [0]:
# Create a column based on current date
curr_date = str(dt.today().strftime('%Y_%m_%d'))
print(curr_date)

df_trx_feats = df_trx_feats.withColumn("PULL_DT", F.lit(curr_date))

df_trx_feats.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/kerendia_nba_3_hcp_model_trx_features_w_provider_id_for_inference")

In [0]:
print("END OF NOTEBOOK")